In [ ]:
!pip install kaggle -q
from google.colab import files
files.upload()


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d emmarex/plantdisease --force
!unzip -o -q plantdisease.zip -d data/

In [ ]:
import os
data_dir = "data/PlantVillage"
classes = os.listdir(data_dir)
print(f"Found {len(classes)} classes:")
print(classes)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
sample_class = classes[0]
sample_dir = os.path.join(data_dir, sample_class)
sample_images = os.listdir(sample_dir)[:5]
fig, axes = plt.subplots(1,5, figsize=(15, 3))
for ax, img_name in zip(axes, sample_images):
  img = Image.open(os.path.join(sample_dir, img_name))
  ax.imshow(img)
  ax.set_title(sample_class, fontsize=8)
  ax.axis('off')
plt.show()

In [ ]:
import tensorflow as tf

data_dir = "data/PlantVillage"
img_size = (128, 128)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
from tensorflow.keras import layers, models

num_classes = len(class_names)

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128, 128, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


In [ ]:
model.save('farmsense_model.h5')
print("Model saved successfully!")

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Loss')

plt.show()

In [ ]:
import numpy as np

# Pick a random image from validation data
for images, labels in val_ds.take(1):
    img = images[0].numpy().astype("uint8")
    true_label = class_names[labels[0]]
    break

# Predict
img_array = tf.expand_dims(images[0], 0)  # add batch dimension
predictions = model.predict(img_array)
predicted_class = class_names[np.argmax(predictions[0])]
confidence = np.max(predictions[0]) * 100

# Show result
plt.imshow(img)
plt.title(f"True: {true_label}\nPredicted: {predicted_class} ({confidence:.1f}%)")
plt.axis('off')
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('farmsense_model.h5', '/content/drive/MyDrive/farmsense_model.h5')
print("Model saved to Google Drive!")

In [ ]:
!pip install gradio -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model('/content/drive/MyDrive/farmsense_model.h5', compile=False)
print("Model loaded successfully!")

In [ ]:
import numpy as np

class_names = ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight',
 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight',
 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot',
 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot',
 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']

def predict(img):
    img = tf.image.resize(img, (128, 128))
    img = tf.expand_dims(img, 0)
    preds = model.predict(img)[0]
    return {class_names[i]: float(preds[i]) for i in range(len(class_names))}

print("Prediction function ready!")

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(label="Upload a leaf photo"),
    outputs=gr.Label(num_top_classes=3, label="Prediction"),
    title="FarmSense AI - Crop Disease Detection"
)
demo.launch(share=True)